# 04. mDeBERTa-v3 base

DeBERTa-v3 changes two things against BERT/RoBERTa: attention looks at content and relative
position separately (disentangled attention), and pre-training is ELECTRA-style replaced-token
detection instead of masked language modeling, which learns from every token, not only the 15%
masked ones. mDeBERTa-v3 is the multilingual version, trained on CC100 like XLM-R, with the same
size as XLM-R base (12 layers, hidden 768).

The data and training settings come from `ner/recipes.py` (decided in notebooks 01 and 02); here we
only change what is specific to this model:

- **lr 3e-5 or 5e-5**: DeBERTa models often prefer a slightly higher learning rate than RoBERTa.
- Precision: on a T4 the run uses fp16. DeBERTa-v3 has been reported to overflow in fp16; if the
  training loss becomes `nan`, add `precision="fp32"` to the overrides (about 2x slower).

Steps:

1. **Learning rate on fold 0.** Two values (stage 1; stage 2 scales with it, as in the recipe); pick
   the one with the higher `half_B` F1.
2. **Folds 1 and 2** with that learning rate. The mean `cv` score over the three folds is the
   cross-validation score of this model, on the same folds as every other model and the Assignment 3
   RNNs.
3. **Fold ensemble.** The three fold models' probabilities on the test posts are averaged. That gives
   the scores on half B and the gold posts, and the Kaggle submission of this model.

On a T4 one fold takes about 40-60 minutes (the `minutes` column shows the measured time).

### Setup

On Colab (T4 or better): clones the repo, installs W&B and keeps every run on Drive in
`dl-a4-ner/runs`, so a disconnect loses at most the run in progress (finished runs are loaded, not
retrained). Teammates: put a shortcut to the shared `dl-a4-ner` folder in your My Drive, so all runs
land in one place. Set `WANDB_ENTITY` to the team's W&B entity to log into one shared project.
Locally the notebook runs from the repo root and writes to `transformers/runs`. To check the whole
notebook in a few minutes before using GPU time, set the environment variable `NER_SMOKE=1`: every run
shrinks to a few hundred texts and goes to `transformers/runs_smoke`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/koalla05/deep-learning-assignment4.git"
WANDB_PROJECT = "dl-a4-ner"  # None switches W&B off
WANDB_ENTITY = None  # the team's W&B entity, so all teammates log into one project

if "google.colab" in sys.modules:
    repo = Path("/content/deep-learning-assignment4")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/dl-a4-ner/runs")
else:
    if Path.cwd().name == "transformers":
        os.chdir("..")
    OUT = Path("transformers/runs")
sys.path.insert(0, os.getcwd())
if os.environ.get("NER_SMOKE"):  # quick end-to-end check with tiny runs, see ner.train.smoke
    OUT, WANDB_PROJECT = Path("transformers/runs_smoke"), None
OUT.mkdir(parents=True, exist_ok=True)
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
if WANDB_PROJECT:
    import wandb

    wandb.login()

In [ ]:
import dataclasses

from ner import plots, recipes
from ner.cv import cv_table, fold_ensemble
from ner.train import RunConfig, load_run, run_or_load

PREFIX = "04-mdeberta"
RECIPE = RunConfig(
    name="recipe", data=recipes.DATA, **(recipes.STAGE2 or {}),
    **{**recipes.OPTIMIZATION, **{}},
    model="microsoft/mdeberta-v3-base", tags=("04-mdeberta",),
)
print(RECIPE)


def run_name(lr, fold):
    return f"{PREFIX}-lr{lr:g}-fold{fold}"


def train(lr, fold):
    stage2_lr = lr * RECIPE.stage2_lr / RECIPE.learning_rate if RECIPE.stage2_lr else None  # same ratio as the recipe
    cfg = dataclasses.replace(RECIPE, name=run_name(lr, fold), learning_rate=lr, stage2_lr=stage2_lr, fold=fold)
    return run_or_load(cfg, OUT, WANDB_PROJECT)

## 1. Learning rate on fold 0

In [ ]:
LEARNING_RATES = [3e-5, 5e-5]
for lr in LEARNING_RATES:
    train(lr, fold=0)

In [ ]:
lr_runs = {f"lr {lr:g}": load_run(OUT, run_name(lr, 0)) for lr in LEARNING_RATES if (OUT / run_name(lr, 0) / "metrics.json").exists()}
plots.results_table(lr_runs, leaderboard="transformers/leaderboard.csv").round(4)

In [ ]:
plots.plot_curves(lr_runs, title="mDeBERTa-v3: learning rate on fold 0")

## 2. Folds 1 and 2

Set `BEST_LR` from the table above (by `half_B`).

In [ ]:
BEST_LR = LEARNING_RATES[0]
for fold in (1, 2):
    train(BEST_LR, fold=fold)

In [ ]:
FOLD_RUNS = [run_name(BEST_LR, k) for k in range(3)]
cv_table(OUT, FOLD_RUNS).round(4)

## 3. Fold ensemble and submission

In [ ]:
ensemble_metrics, sub = fold_ensemble(OUT, FOLD_RUNS)
sub.to_csv(OUT / f"{PREFIX}-fold-ensemble-submission.csv", index=False)
print("submission:", OUT / f"{PREFIX}-fold-ensemble-submission.csv")
ensemble_metrics

In [ ]:
best = load_run(OUT, FOLD_RUNS[0])
plots.plot_curves({"fold 0": best}, title="mDeBERTa-v3, fold 0")
plots.plot_grad_flow(best, title="mDeBERTa-v3, fold 0")

## Conclusions

- CV (mean of `cv` over folds) and its spread, `half_B`, `gold26` and the public leaderboard
  (add the submission's score to `transformers/leaderboard.csv`).
- Compared with XLM-R base (notebook 02 baseline): is this model worth its cost?
- Is the model adequate? Check that the training loss went down smoothly, that held-out loss and F1
  didn't get worse in the last epoch (overfitting), that the gradient norms of the lower layers are not
  zero (they learn) and not exploding, and that CV and `half_B` agree with the leaderboard in order.